# Notebook goals

Combine all datasets on disk into one AnnData object

# imports

In [1]:
# Built-in imports
from glob import glob
from pathlib import Path

# Dependency imports
from anndata.experimental import concat_on_disk
import scanpy as sc
import h5py
import pandas as pd 
import re 
from dotenv import dotenv_values

In [ ]:
# Project-wide Constants
CONFIG = dotenv_values("config.env") # assuming kernel located at the root of the directory
DATA_ROOT_DIR = Path(CONFIG['DATADIR'])
RCPARAMS = CONFIG['MPLSTYLE']

In [3]:
# Notebook-specific constants; automatically derived depending on user input above

DATADIR = DATA_ROOT_DIR / "2b_clean"

In [4]:
adata_fps = sorted(glob(Path(DATADIR, 'SC*.h5ad').as_posix()))
adata_fps

['/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-01.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-02.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-03.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-04.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-05.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-06.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-07.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-08.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-09.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-10.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-11.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-12.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-13.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-14.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-15.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/SC1-16.h5ad',
 '/REDACTED-path/HypoMap3D-data/v0.9/2b_

### Concatenate all AnnData objects on-disk 

roughly 20 min

In [5]:
concat_fp = Path(DATADIR, f'concat.h5ad')
concat_on_disk(adata_fps, concat_fp, index_unique=None)

### Re-attach .Var and delete falsly joined .obsm

In [6]:
with h5py.File(adata_fps[0], "r") as template, h5py.File(concat_fp, "a") as concat:
    if "/var" in template:
        if "/var" in concat:
            del concat["/var"]
        template.copy("/var", concat)
    else:
        print("Source file does not contain a /var group.")
        
    if "/obsm" in concat:
        del concat["/obsm"]

### Check the resulting AnnData object

In [7]:
concat_fp = Path(DATADIR, f'concat.h5ad')
combined = sc.read_h5ad(concat_fp,)
combined

AnnData object with n_obs × n_vars = 2030770 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'manually_removed'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene'